In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from scipy.stats import entropy

In [2]:
import torch

device = torch.device("cpu")
print("Using device:", device)

Using device: cpu


In [3]:
X_train = pd.read_csv("X_train.csv")
Y_train = pd.read_csv("Y_train.csv")

df = X_train.merge(Y_train, on="id")

In [13]:
# Lấy toàn bộ action IDs
all_actions = df.iloc[:, 1:-6].values.flatten()
all_actions = all_actions[~pd.isna(all_actions) & ~np.isinf(all_actions)] # Added filtering for Inf values

unique_actions = sorted(set(all_actions))
action2idx = {a: i+1 for i, a in enumerate(unique_actions)}  # 0 = PAD
idx2action = {v: k for k, v in action2idx.items()}

In [14]:
MAX_LEN = 24

def process_sequence(row):
    actions = row.values
    actions = actions[~pd.isna(actions)]

    # Map
    mapped = [action2idx[a] for a in actions]

    seq_length = len(mapped)

    # Truncate
    if len(mapped) > MAX_LEN:
        mapped = mapped[-MAX_LEN:]

    # Pad
    padded = mapped + [0]*(MAX_LEN - len(mapped))

    # Mask
    mask = [1 if x != 0 else 0 for x in padded]

    return padded, mask, seq_length

In [15]:
def compute_aux_features(row):
    actions = row.values
    actions = actions[~pd.isna(actions)]

    length = len(actions)
    unique_count = len(set(actions))

    unique_ratio = unique_count / length if length > 0 else 0

    counter = Counter(actions)
    probs = np.array(list(counter.values())) / length
    ent = entropy(probs)

    repetition_ratio = 1 - unique_ratio

    first_action = actions[0] if length > 0 else 0
    last_action = actions[-1] if length > 0 else 0

    first_action = action2idx.get(first_action, 0)
    last_action = action2idx.get(last_action, 0)

    return np.array([
        length,
        unique_ratio,
        repetition_ratio,
        ent,
        first_action,
        last_action
    ], dtype=np.float32)

In [16]:
class BehaviorDataset(Dataset):
    def __init__(self, df):
        self.sequences = []
        self.masks = []
        self.aux_features = []
        self.labels = []

        for _, row in df.iterrows():
            seq_row = row.iloc[1:-6]
            label_row = row.iloc[-6:].values.astype(np.int64)

            padded, mask, seq_length = process_sequence(seq_row)
            aux = compute_aux_features(seq_row)

            self.sequences.append(padded)
            self.masks.append(mask)
            self.aux_features.append(aux)
            self.labels.append(label_row)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx], dtype=torch.long),
            torch.tensor(self.masks[idx], dtype=torch.float32),
            torch.tensor(self.aux_features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

In [18]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=24):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) *
                             -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

In [17]:
class BehaviorModel(nn.Module):
    def __init__(self, vocab_size, num_classes_list):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size+1, 256, padding_idx=0)
        self.pos_encoding = PositionalEncoding(256)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            dim_feedforward=512,
            dropout=0.3,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.aux_fc = nn.Linear(6, 64)

        self.shared_fc = nn.Linear(256 + 64, 256)

        # Shared branch for attr_1 & attr_4
        self.branch_14 = nn.Linear(256, 128)

        # Output heads
        self.heads = nn.ModuleList([
            nn.Linear(128 if i in [0,3] else 256, num_classes_list[i])
            for i in range(6)
        ])

    def forward(self, seq, mask, aux):

        x = self.embedding(seq)
        x = self.pos_encoding(x)

        x = self.transformer(x, src_key_padding_mask=(mask==0))

        # Global average pooling
        x = (x * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)

        aux = F.relu(self.aux_fc(aux))

        x = torch.cat([x, aux], dim=1)
        x = F.relu(self.shared_fc(x))

        outputs = []
        branch14 = F.relu(self.branch_14(x))

        for i in range(6):
            if i in [0,3]:
                outputs.append(self.heads[i](branch14))
            else:
                outputs.append(self.heads[i](x))

        return outputs

In [19]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

In [20]:
def compute_loss(outputs, labels, loss_fns, weights):
    total = 0
    for i in range(6):
        total += weights[i] * loss_fns[i](outputs[i], labels[:, i])
    return total

In [21]:
def exact_match(outputs, labels):
    preds = [torch.argmax(o, dim=1) for o in outputs]
    preds = torch.stack(preds, dim=1)
    correct = (preds == labels).all(dim=1)
    return correct.float().mean().item()

#Chạy thử model

In [8]:
import torch
import torch.optim as optim
from sklearn.metrics import f1_score
from tqdm import tqdm

##TẠO DATASET & DATALOADER

In [6]:
X_train = pd.read_csv("X_train.csv")
Y_train = pd.read_csv("Y_train.csv")

train_df = X_train.merge(Y_train, on="id")

In [7]:
train_id = train_df['id'].values

In [9]:
X_val = pd.read_csv("X_val.csv")
Y_val = pd.read_csv("Y_val.csv")

val_df = X_val.merge(Y_val, on="id") # Fixed: Changed X_train to X_val

In [10]:
valid_id = val_df['id'].values

In [ ]:
label_encoders = {}
num_classes_list = []

for col in ['attr_1','attr_2','attr_3','attr_4','attr_5','attr_6']:
    # Get sorted unique values
    unique_vals = sorted(train_df[col].unique())

    # Ensure mapping starts at 0
    mapping = {v: i for i, v in enumerate(unique_vals)}
    label_encoders[col] = mapping

    train_df[col] = train_df[col].map(mapping)
    val_df[col]   = val_df[col].map(mapping)

    num_classes_list.append(len(unique_vals))

In [ ]:
for i, col in enumerate(['attr_1','attr_2','attr_3','attr_4','attr_5','attr_6']):
    print(col)
    print("min:", train_df[col].min())
    print("max:", train_df[col].max())
    print("num_classes:", num_classes_list[i])
    print()

attr_1
min: 0
max: 11
num_classes: 12

attr_2
min: 0
max: 30
num_classes: 31

attr_3
min: 0
max: 98
num_classes: 99

attr_4
min: 0
max: 11
num_classes: 12

attr_5
min: 0
max: 30
num_classes: 31

attr_6
min: 0
max: 98
num_classes: 99



In [ ]:
for i, col in enumerate(['attr_1','attr_2','attr_3','attr_4','attr_5','attr_6']):
    print(col)
    print("min:", val_df[col].min())
    print("max:", val_df[col].max())
    print("num_classes:", num_classes_list[i])
    print()

attr_1
min: 0
max: 11
num_classes: 12

attr_2
min: 0
max: 30
num_classes: 31

attr_3
min: 0
max: 98
num_classes: 99

attr_4
min: 0
max: 11
num_classes: 12

attr_5
min: 0
max: 30
num_classes: 31

attr_6
min: 0
max: 98
num_classes: 99



In [22]:
train_dataset = BehaviorDataset(train_df)
val_dataset   = BehaviorDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False)

##KHỞI TẠO MODEL

In [ ]:
num_classes_list = [
    train_df['attr_1'].max() + 1,
    train_df['attr_2'].max() + 1,
    train_df['attr_3'].max() + 1,
    train_df['attr_4'].max() + 1,
    train_df['attr_5'].max() + 1,
    train_df['attr_6'].max() + 1
]

# Re-initialize model with correct class counts and move to GPU
model = BehaviorModel(
    vocab_size=len(action2idx),
    num_classes_list=num_classes_list
).to(device)

##LOSS & OPTIMIZER

In [ ]:
loss_fns = [
    FocalLoss(gamma=2),  # attr_1 (imbalanced)
    nn.CrossEntropyLoss(),
    nn.CrossEntropyLoss(),
    FocalLoss(gamma=2),  # attr_4 (imbalanced)
    nn.CrossEntropyLoss(),
    nn.CrossEntropyLoss()
]

weights = [1.5, 1, 1, 1.5, 1, 1]

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

##TRAINING FUNCTION

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for seq, mask, aux, labels in loader:

        seq = seq.to(device)
        mask = mask.to(device)
        aux = aux.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(seq, mask, aux)

        loss = compute_loss(outputs, labels, loss_fns, weights)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

##VALIDATION FUNCTION

In [ ]:
from sklearn.metrics import f1_score

def evaluate(model, loader):
    model.eval()

    all_preds = [[] for _ in range(6)]
    all_labels = [[] for _ in range(6)]
    exact_correct = 0
    total_samples = 0

    with torch.no_grad():
        for seq, mask, aux, labels in loader:

            seq = seq.to(device)
            mask = mask.to(device)
            aux = aux.to(device)
            labels = labels.to(device)

            outputs = model(seq, mask, aux)

            preds = [torch.argmax(o, dim=1) for o in outputs]

            # Exact Match
            stacked_preds = torch.stack(preds, dim=1)
            correct = (stacked_preds == labels).all(dim=1)
            exact_correct += correct.sum().item()
            total_samples += labels.size(0)

            for i in range(6):
                all_preds[i].extend(preds[i].cpu().numpy())
                all_labels[i].extend(labels[:, i].cpu().numpy())

    exact_match = exact_correct / total_samples

    macro_f1 = []
    for i in range(6):
        f1 = f1_score(all_labels[i], all_preds[i], average="macro")
        macro_f1.append(f1)

    return exact_match, macro_f1, preds

##TRAIN LOOP HOÀN CHỈNH

In [ ]:
best_exact = 0
EPOCHS = 20

for epoch in range(EPOCHS):

    print(f"\n===== Epoch {epoch+1} =====")

    train_loss = train_one_epoch(model, train_loader)

    exact_match, macro_f1 = evaluate(model, val_loader)

    scheduler.step()

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Exact Match: {exact_match:.4f}")
    print(f"Macro F1 per attr: {macro_f1}")

    if exact_match > best_exact:
        best_exact = exact_match
        torch.save(model.state_dict(), "best_model_cpu.pth")
        print("Saved best model!")


===== Epoch 1 =====


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Train Loss: 12.4038
Exact Match: 0.0918
Macro F1 per attr: [0.5854022640081816, 0.649136536102349, 0.1910986991196027, 0.5853904742420647, 0.6912762876454315, 0.42998453765049227]
Saved best model!

===== Epoch 2 =====
Train Loss: 4.6685
Exact Match: 0.4740
Macro F1 per attr: [0.659096710109048, 0.8316208156931096, 0.7316509023938607, 0.6690124629763425, 0.7795116018658407, 0.8045282186745304]
Saved best model!

===== Epoch 3 =====
Train Loss: 1.8349
Exact Match: 0.6529
Macro F1 per attr: [0.770473552606867, 0.8981046946832948, 0.8876754080207464, 0.7395568151450819, 0.8603132506724411, 0.8989482741997312]
Saved best model!

===== Epoch 4 =====
Train Loss: 0.8979
Exact Match: 0.7500
Macro F1 per attr: [0.7967630897040019, 0.9470088181831461, 0.9274877709291754, 0.771235945225259, 0.9074516156585963, 0.9539716862491268]
Saved best model!

===== Epoch 5 =====
Train Loss: 0.5205
Exact Match: 0.8382
Macro F1 per attr: [0.7997083021023984, 0.9656788560025307, 0.9571871803053863, 0.802858968

KeyboardInterrupt: 

##LOAD BEST MODEL & FINAL EVAL

In [ ]:
model.load_state_dict(torch.load("best_model_cpu.pth"))

exact_match, macro_f1 = evaluate(model, val_loader)

print("\nFINAL VALIDATION RESULT")
print("Exact Match:", exact_match)
print("Macro F1:", macro_f1)


FINAL VALIDATION RESULT
Exact Match: 0.94375
Macro F1: [0.880138793843437, 0.9899755031567816, 0.9815586087116421, 0.8896152481141608, 0.9839961679171648, 0.9928868667230719]


In [ ]:
model.load_state_dict(torch.load("best_model_cpu_8epoch.pth"))

exact_match, macro_f1 = evaluate(model, val_loader)

print("\nFINAL VALIDATION RESULT")
print("Exact Match:", exact_match)
print("Macro F1:", macro_f1)


FINAL VALIDATION RESULT
Exact Match: 0.9218055555555555
Macro F1: [0.8814495800399788, 0.9846129451906771, 0.9772122905978865, 0.8821997589445774, 0.9774861894058912, 0.9892868829905637]


In [ ]:
model.load_state_dict(torch.load("best_model_cpu_10epoch.pth"))

exact_match, macro_f1 = evaluate(model, val_loader)

print("\nFINAL VALIDATION RESULT")
print("Exact Match:", exact_match)
print("Macro F1:", macro_f1)


FINAL VALIDATION RESULT
Exact Match: 0.9420833333333334
Macro F1: [0.8882195503501046, 0.9907050938472666, 0.9815077594502726, 0.8932736074702222, 0.9834337102878598, 0.9920793223451071]


In [ ]:
model.load_state_dict(torch.load("best_model_cpu_12epoch.pth"))

exact_match, macro_f1, preds = evaluate(model, val_loader)

print("\nFINAL VALIDATION RESULT")
print("Exact Match:", exact_match)
print("Macro F1:", macro_f1)


FINAL VALIDATION RESULT
Exact Match: 0.9422222222222222
Macro F1: [0.8825961731195543, 0.9903374556421201, 0.9804967558101256, 0.8873276114954205, 0.9833240796189359, 0.9927797614265632]


In [ ]:
print(preds)

[tensor([9, 1, 0, 0, 6, 6, 5, 5, 3, 0, 9, 7, 1, 0, 6, 0, 6, 9, 6, 3, 9, 9, 6, 3,
        0, 0, 9, 6, 0, 6, 6, 6]), tensor([ 0,  0,  0,  0, 23,  0,  5,  0,  0,  0, 13,  8,  0,  0,  0,  0, 20, 25,
        21,  0,  0,  1,  0, 24,  0,  0, 15,  2,  0,  0,  3, 23]), tensor([79, 13, 18, 16, 29, 60, 32, 13, 87, 35, 84, 26,  6, 22, 94, 70, 59, 37,
        12, 26, 53, 26,  3, 63, 63, 36, 15, 50, 53, 81, 69, 76]), tensor([6, 6, 0, 0, 3, 5, 1, 3, 9, 0, 9, 1, 3, 0, 5, 0, 9, 5, 3, 6, 9, 9, 5, 5,
        0, 0, 6, 1, 0, 1, 9, 9]), tensor([ 0,  0,  0,  0, 23,  0, 30,  0,  0,  0, 23,  1,  0,  0,  0,  0,  9, 25,
        11,  0,  0, 18,  0, 15,  0,  0, 30, 24,  0,  0, 19,  6]), tensor([20, 78, 60, 84, 65, 88, 76, 28, 62, 32, 34, 94, 26, 15, 44, 70, 83, 72,
        49, 78, 37, 67, 16, 63, 19, 86, 16, 98, 16, 18,  6, 49])]


##HÀM LẤY FULL PREDICTIONS

In [11]:
def get_predictions(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for seq, mask, aux, labels in loader:

            seq = seq.to(device)
            mask = mask.to(device)
            aux = aux.to(device)
            labels = labels.to(device)

            outputs = model(seq, mask, aux)

            preds = [torch.argmax(o, dim=1) for o in outputs]
            stacked_preds = torch.stack(preds, dim=1)

            all_preds.append(stacked_preds.cpu())
            all_labels.append(labels.cpu())

    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()

    # Exact Match
    exact_match = (all_preds == all_labels).all(axis=1).mean()

    # Macro F1 từng thuộc tính
    macro_f1 = []
    for i in range(6):
        f1 = f1_score(all_labels[:, i], all_preds[:, i], average="macro")
        macro_f1.append(f1)

    return all_preds, all_labels, exact_match, macro_f1

In [ ]:
train_preds, train_labels, train_exact, train_f1 = get_predictions(model, train_loader)

print("===== TRAIN RESULT =====")
print("Exact Match:", train_exact)
print("Macro F1:", train_f1)
print("Shape preds:", train_preds.shape)

===== TRAIN RESULT =====
Exact Match: 0.9951764705882353
Macro F1: [0.9628878827003451, 0.9996515431397454, 0.9991601677816623, 0.9685447620900067, 0.9992393465501063, 0.9990898878526996]
Shape preds: (51000, 6)


In [ ]:
val_preds, val_labels, val_exact, val_f1 = get_predictions(model, val_loader)

print("===== VALID RESULT =====")
print("Exact Match:", val_exact)
print("Macro F1:", val_f1)
print("Shape preds:", val_preds.shape)

===== VALID RESULT =====
Exact Match: 0.94375
Macro F1: [0.880138793843437, 0.9899755031567816, 0.9815586087116421, 0.8896152481141608, 0.9839961679171648, 0.9928868667230719]
Shape preds: (7200, 6)


In [ ]:
train_result_df = pd.DataFrame(train_preds, columns=[
    "pred_attr_1",
    "pred_attr_2",
    "pred_attr_3",
    "pred_attr_4",
    "pred_attr_5",
    "pred_attr_6"
])


for i in range(6):

    train_result_df[f"true_attr_{i+1}"] = train_labels[:, i]

train_result_df.to_csv("train_predictions.csv", index=True)

In [ ]:
val_result_df = pd.DataFrame(val_preds, columns=[
    "pred_attr_1",
    "pred_attr_2",
    "pred_attr_3",
    "pred_attr_4",
    "pred_attr_5",
    "pred_attr_6"
])

for i in range(6):
    val_result_df[f"true_attr_{i+1}"] = val_labels[:, i]

val_result_df.to_csv("validation_predictions.csv", index=False)

##PREDICT TEST SET (KAGGLE SUBMISSION)

In [ ]:
class BehaviorTestDataset(Dataset):
    def __init__(self, df):
        self.ids = df["id"].values
        self.sequences = []
        self.masks = []
        self.aux_features = []

        for _, row in df.iterrows():
            seq_row = row.iloc[1:]

            padded, mask, _ = process_sequence(seq_row)
            aux = compute_aux_features(seq_row)

            self.sequences.append(padded)
            self.masks.append(mask)
            self.aux_features.append(aux)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx], dtype=torch.long),
            torch.tensor(self.masks[idx], dtype=torch.float32),
            torch.tensor(self.aux_features[idx], dtype=torch.float32),
            self.ids[idx]
        )

In [ ]:
X_test = pd.read_csv("X_test.csv")

test_dataset = BehaviorTestDataset(X_test)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [ ]:
BehaviorTestDataset

__main__.BehaviorTestDataset

In [ ]:
reverse_label_encoders = {}

for col in label_encoders:
    reverse_label_encoders[col] = {
        v: k for k, v in label_encoders[col].items()
    }

In [ ]:
device = torch.device("cpu")

model = BehaviorModel(
    vocab_size=len(action2idx),
    num_classes_list=num_classes_list
).to(device)

model.load_state_dict(torch.load("best_model_cpu.pth", map_location=device))
model.eval()

BehaviorModel(
  (embedding): Embedding(255, 256, padding_idx=0)
  (pos_encoding): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (aux_fc): Linear(in_features=6, out_features=64, bias=True)
  (shared_fc): Linear(in_features=320, out_features=256, bias=True)
  (branch_14): Linear(in_features=256, out_features=128, 

In [ ]:
def predict_test(model, loader):

    all_ids = []
    all_preds = []

    with torch.no_grad():
        for seq, mask, aux, ids in loader:

            seq = seq.to(device)
            mask = mask.to(device)
            aux = aux.to(device)

            outputs = model(seq, mask, aux)
            preds = [torch.argmax(o, dim=1) for o in outputs]

            stacked = torch.stack(preds, dim=1)  # (batch, 6)

            all_preds.extend(stacked.cpu().numpy())
            all_ids.extend(ids)

    return all_ids, all_preds

In [ ]:
ids, predictions = predict_test(model, test_loader)

converted_preds = []

for row in predictions:
    converted_row = []

    for i in range(6):
        original_label = reverse_label_encoders[i][int(row[i])]
        converted_row.append(original_label)

    converted_preds.append(converted_row)

converted_preds = np.array(converted_preds, dtype=np.uint16)

KeyError: 0

In [ ]:
submission_df = pd.DataFrame({
    "id": ids,
    "attr_1": converted_preds[:, 0],
    "attr_2": converted_preds[:, 1],
    "attr_3": converted_preds[:, 2],
    "attr_4": converted_preds[:, 3],
    "attr_5": converted_preds[:, 4],
    "attr_6": converted_preds[:, 5],
})

submission_df.to_csv("team_name.csv", index=False)

print("Submission file saved!")

In [ ]:
submission_df.iloc[:, 1:] = np.array(predictions).astype(np.uint16)
submission_df.to_csv("team_name.csv", index=False)